# run goodness of fit test for degree distributions with the null being that they are binomially distributed

In [ ]:
import numpy as np
import os
from scipy import stats
from collections import defaultdict
import csv


In [7]:
# Directory with degree sequence files
save_dir = './degree_sequences_np/'
data_dir = './degree_sequences_np/inter'

# Store results by network type and size
results = defaultdict(list)

alpha = 0.05  # significance level

def fit_binomial_mle(degrees, n_trials = None):
    """Fit binomial distribution using MLE 
    If nothing proveded for n_trials, then estimate both n and p. Search over n from max(degrees) to 2*max(degrees) and find best fit.
    Otherwise just return the provided n_trials and the estimate for p"""
    max_deg = degrees.max()
    mean_deg = degrees.mean()

    if n_trials is None: # estimate n and p
        best_n = max_deg
        best_loglik = -np.inf
        
        # Search over possible n values
        for n in range(max_deg, max(2 * max_deg, max_deg + 100)):
            p = mean_deg / n
            if p <= 0 or p >= 1:
                continue
            # Log-likelihood
            loglik = np.sum(stats.binom.logpmf(degrees, n, p))
            if loglik > best_loglik:
                best_loglik = loglik
                best_n = n
    else:  
        best_n = n_trials
    
    best_p = mean_deg / best_n
    return best_n, best_p


for filename in os.listdir(data_dir):
    if not filename.endswith('.csv'):
        continue
    
    # Parse network type from filename
    if '_power_' in filename:
        net_type = 'power'
    elif '_rand_' in filename:
        net_type = 'rand'
    elif '_exp_' in filename:
        net_type = 'exp'
    else:
        continue
    
    # Parse num species from filename
    if '1000_' in filename:
        n_spec = 1000
    elif '100_' in filename:
        n_spec = 100
    else:
        continue

    # Parse interaction strength
    if 'Fmax0.3333_' in filename:
        fmax = '0.3333'
    elif 'Fmax0.2_' in filename:
        fmax = '0.2'
    elif 'Fmax0.1_' in filename:
        fmax = '0.1'
    else:
        continue
    
    key = (net_type, str(n_spec), fmax)
    
    # Load degree sequence (skip header, take second column)
    degrees = np.loadtxt(os.path.join(data_dir, filename), delimiter=',', skiprows=1, usecols=1, dtype=int)
    n_nodes = len(degrees)
    
    # Binomial MLE for both n and p
    # changed my mind, not estimating n, just estimating p, which is then an analytical form
    # n_trials, p_est = fit_binomial_mle(degrees)
    n_trials, p_est = fit_binomial_mle(degrees, n_trials = n_spec - 1)
    
    # Chi-squared goodness-of-fit test
    max_deg = degrees.max()
    observed_counts = np.bincount(degrees, minlength=max_deg + 1)
    
    # Expected counts from binomial
    expected_probs = stats.binom.pmf(np.arange(max_deg + 1), n_trials, p_est)
    expected_counts = expected_probs * n_nodes
    expected_counts = expected_counts * (observed_counts.sum() / expected_counts.sum())
    
    # Combine bins with expected count < 5
    obs_binned = []
    exp_binned = []
    obs_acc, exp_acc = 0, 0
    
    for obs, exp in zip(observed_counts, expected_counts):
        obs_acc += obs
        exp_acc += exp
        if exp_acc >= 5:
            obs_binned.append(obs_acc)
            exp_binned.append(exp_acc)
            obs_acc, exp_acc = 0, 0
    
    if obs_acc > 0 or exp_acc > 0:
        if len(exp_binned) > 0:
            obs_binned[-1] += obs_acc
            exp_binned[-1] += exp_acc
        else:
            obs_binned.append(obs_acc)
            exp_binned.append(exp_acc)
    
    if len(obs_binned) >= 2:
        # ddof = 2  # estimated both n and p
        ddof = 1 # estimated just p
        chi2, pval = stats.chisquare(obs_binned, exp_binned, ddof=ddof)
        reject_null = pval < alpha
    else:
        pval = np.nan
        reject_null = np.nan
    
    results[key].append({
        'filename': filename,
        'pval': pval,
        'reject': reject_null
    })

# Summary: rejection rate by network type and size
print("INTERACTION NETWORKS")
print("Rejection rate of binomial null hypothesis (alpha=0.05):\n")
for n_spec in ['100', '1000']:
    print(f"--- {n_spec} nodes ---")
    for fmax in ['0.3333', '0.2', '0.1']:
        print(f"--- {fmax} max interact factor ---")
        for net_type in ['rand', 'exp', 'power']:
            key = (net_type, n_spec, fmax)
            if key in results:
                rejections = [r['reject'] for r in results[key] if not np.isnan(r['reject'])]
                rate = np.mean(rejections) if rejections else 0
                print(f"{net_type:6s}: {rate*100:5.1f}% rejected ({sum(rejections)}/{len(rejections)} networks)")
    print()


INTERACTION NETWORKS
Rejection rate of binomial null hypothesis (alpha=0.05):

--- 100 nodes ---
--- 0.3333 max interact factor ---
rand  :   4.0% rejected (2/50 networks)
exp   : 100.0% rejected (50/50 networks)
power : 100.0% rejected (50/50 networks)
--- 0.2 max interact factor ---
rand  :   4.0% rejected (2/50 networks)
exp   : 100.0% rejected (50/50 networks)
power : 100.0% rejected (50/50 networks)
--- 0.1 max interact factor ---
rand  :   6.0% rejected (3/50 networks)
exp   : 100.0% rejected (50/50 networks)
power : 100.0% rejected (50/50 networks)

--- 1000 nodes ---
--- 0.3333 max interact factor ---
rand  :  10.0% rejected (5/50 networks)
exp   : 100.0% rejected (50/50 networks)
power : 100.0% rejected (50/50 networks)
--- 0.2 max interact factor ---
rand  :   6.0% rejected (3/50 networks)
exp   : 100.0% rejected (50/50 networks)
power : 100.0% rejected (50/50 networks)
--- 0.1 max interact factor ---
rand  :   4.0% rejected (2/50 networks)
exp   : 100.0% rejected (50/50 netw

In [8]:
# Write detailed interaction results to CSV
output_file = os.path.join(save_dir, 'interact_binomial_test_results.csv')

with open(output_file, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header
    writer.writerow(['network_type', 'network_size', 'fmax', 'filename', 'p_value', 'reject_null'])
    
    # Write data rows
    for key, result_list in results.items():
        net_type, n_spec, fmax = key
        for result in result_list:
            writer.writerow([
                net_type,
                n_spec,
                fmax,
                result['filename'],
                result['pval'],
                result['reject']
            ])

print(f"Results saved to: {output_file}")

# Write summary statistics to a separate CSV
summary_file = os.path.join(save_dir, 'interact_rejection_rate_summary.csv')

with open(summary_file, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header
    writer.writerow(['network_type', 'network_size', 'fmax', 'rejection_rate', 'n_rejected', 'n_total'])
    
    # Write summary data
    for n_spec in ['100', '1000']:
        for fmax in ['0.3333', '0.2', '0.1']:
            for net_type in ['rand', 'exp', 'power']:
                key = (net_type, n_spec, fmax)
                if key in results:
                    rejections = [r['reject'] for r in results[key] if not np.isnan(r['reject'])]
                    if rejections:
                        rate = np.mean(rejections)
                        n_rejected = sum(rejections)
                        n_total = len(rejections)
                    else:
                        rate = 0
                        n_rejected = 0
                        n_total = 0
                    
                    writer.writerow([net_type, n_spec, fmax, rate, n_rejected, n_total])

print(f"Summary statistics saved to: {summary_file}")

Results saved to: ./degree_sequences_np/interact_binomial_test_results.csv
Summary statistics saved to: ./degree_sequences_np/interact_rejection_rate_summary.csv


In [9]:
# Directory with co-occurrence degree sequence files
data_dir_cooc = './degree_sequences_np/cooc'

# Store results by network type and size
results_cooc = defaultdict(list)

for filename in os.listdir(data_dir_cooc):
    if not filename.endswith('.csv'):
        continue
    
    # Parse network type from filename
    if '_power_' in filename:
        net_type = 'power'
    elif '_rand_' in filename:
        net_type = 'rand'
    elif '_exp_' in filename:
        net_type = 'exp'
    else:
        continue
    
    # Parse num species from filename
    if '1000_' in filename:
        n_spec = 1000
    elif '100_' in filename:
        n_spec = 100
    else:
        continue

    # Parse interaction strength
    if 'Fmax0.3333_' in filename:
        fmax = '0.3333'
    elif 'Fmax0.2_' in filename:
        fmax = '0.2'
    elif 'Fmax0.1_' in filename:
        fmax = '0.1'
    else:
        continue
    
    key = (net_type, str(n_spec), fmax)
    
    # Load degree sequence (skip header, take second column)
    degrees = np.loadtxt(os.path.join(data_dir_cooc, filename), delimiter=',', skiprows=1, usecols=1, dtype=int)
    n_nodes = len(degrees)
    
    # Binomial MLE for both n and p
    # n_trials, p_est = fit_binomial_mle(degrees)
    n_trials, p_est = fit_binomial_mle(degrees, n_trials = n_spec - 1)
    
    # Chi-squared goodness-of-fit test
    max_deg = degrees.max()
    observed_counts = np.bincount(degrees, minlength=max_deg + 1)
    
    # Expected counts from binomial
    expected_probs = stats.binom.pmf(np.arange(max_deg + 1), n_trials, p_est)
    expected_counts = expected_probs * n_nodes
    expected_counts = expected_counts * (observed_counts.sum() / expected_counts.sum())
    
    # Combine bins with expected count < 5
    obs_binned = []
    exp_binned = []
    obs_acc, exp_acc = 0, 0
    
    for obs, exp in zip(observed_counts, expected_counts):
        obs_acc += obs
        exp_acc += exp
        if exp_acc >= 5:
            obs_binned.append(obs_acc)
            exp_binned.append(exp_acc)
            obs_acc, exp_acc = 0, 0
    
    if obs_acc > 0 or exp_acc > 0:
        if len(exp_binned) > 0:
            obs_binned[-1] += obs_acc
            exp_binned[-1] += exp_acc
        else:
            obs_binned.append(obs_acc)
            exp_binned.append(exp_acc)
    
    if len(obs_binned) >= 2:
        ddof = 2  # estimated both n and p
        chi2, pval = stats.chisquare(obs_binned, exp_binned, ddof=ddof)
        reject_null = pval < alpha
    else:
        pval = np.nan
        reject_null = np.nan
    
    results_cooc[key].append({
        'filename': filename,
        'pval': pval,
        'reject': reject_null
    })

# Summary: rejection rate by network type, size, and cutoff alpha
print("CO-OCCURRENCE NETWORKS")
print("Rejection rate of binomial null hypothesis (alpha=0.05):\n")
for n_spec in ['100', '1000']:
    for fmax in ['0.3333', '0.2', '0.1']:
        print(f"--- {n_spec} nodes, fmax={fmax} ---")
        for net_type in ['rand', 'exp', 'power']:
            key = (net_type, n_spec, fmax)
            if key in results_cooc:
                rejections = [r['reject'] for r in results_cooc[key] if not np.isnan(r['reject'])]
                rate = np.mean(rejections) if rejections else 0
                print(f"{net_type:6s}: {rate*100:5.1f}% rejected ({sum(rejections)}/{len(rejections)} networks)")
        print()


CO-OCCURRENCE NETWORKS
Rejection rate of binomial null hypothesis (alpha=0.05):

--- 100 nodes, fmax=0.3333 ---
rand  :  14.0% rejected (7/50 networks)
exp   : 100.0% rejected (50/50 networks)
power :  56.0% rejected (28/50 networks)

--- 100 nodes, fmax=0.2 ---
rand  :  12.0% rejected (6/50 networks)
exp   :  42.0% rejected (21/50 networks)
power :  30.0% rejected (15/50 networks)

--- 100 nodes, fmax=0.1 ---
rand  :  10.0% rejected (5/50 networks)
exp   :  20.0% rejected (10/50 networks)
power :  12.0% rejected (6/50 networks)

--- 1000 nodes, fmax=0.3333 ---
rand  :  16.0% rejected (8/50 networks)
exp   : 100.0% rejected (50/50 networks)
power :  98.0% rejected (49/50 networks)

--- 1000 nodes, fmax=0.2 ---
rand  :   4.0% rejected (2/50 networks)
exp   :  94.0% rejected (47/50 networks)
power :  78.0% rejected (39/50 networks)

--- 1000 nodes, fmax=0.1 ---
rand  :   4.0% rejected (2/50 networks)
exp   :   6.0% rejected (3/50 networks)
power :  14.0% rejected (7/50 networks)



In [10]:
# Write detailed results to CSV
output_file = os.path.join(save_dir, 'cooc_binomial_test_results.csv')

with open(output_file, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header
    writer.writerow(['network_type', 'network_size', 'fmax', 'filename', 'p_value', 'reject_null'])
    
    # Write data rows
    for key, result_list in results_cooc.items():
        net_type, n_spec, fmax = key
        for result in result_list:
            writer.writerow([
                net_type,
                n_spec,
                fmax,
                result['filename'],
                result['pval'],
                result['reject']
            ])

print(f"Results saved to: {output_file}")

# Write summary statistics to a separate CSV
summary_file = os.path.join(save_dir, 'cooc_rejection_rate_summary.csv')

with open(summary_file, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header
    writer.writerow(['network_type', 'network_size', 'fmax', 'rejection_rate', 'n_rejected', 'n_total'])
    
    # Write summary data
    for n_spec in ['100', '1000']:
        for fmax in ['0.3333', '0.2', '0.1']:
            for net_type in ['rand', 'exp', 'power']:
                key = (net_type, n_spec, fmax)
                if key in results_cooc:
                    rejections = [r['reject'] for r in results_cooc[key] if not np.isnan(r['reject'])]
                    if rejections:
                        rate = np.mean(rejections)
                        n_rejected = sum(rejections)
                        n_total = len(rejections)
                    else:
                        rate = 0
                        n_rejected = 0
                        n_total = 0
                    
                    writer.writerow([net_type, n_spec, fmax, rate, n_rejected, n_total])

print(f"Summary statistics saved to: {summary_file}")

Results saved to: ./degree_sequences_np/cooc_binomial_test_results.csv
Summary statistics saved to: ./degree_sequences_np/cooc_rejection_rate_summary.csv


In [11]:
# Test: verify rejection rate is ~0.05 when data is truly binomial
def test_binomial_rejection_rate(n_true=50, p_true=0.3, sample_size=100, n_simulations=200, alpha=0.05):
    """Generate binomial samples and check that rejection rate ≈ alpha."""
    rejections = []
    
    for _ in range(n_simulations):
        # Generate truly binomial data
        degrees = np.random.binomial(n_true, p_true, size=sample_size)
        
        # Fit binomial MLE
        n_est, p_est = fit_binomial_mle(degrees)
        
        # Chi-squared goodness-of-fit test
        max_deg = degrees.max()
        observed_counts = np.bincount(degrees, minlength=max_deg + 1)
        
        expected_probs = stats.binom.pmf(np.arange(max_deg + 1), n_est, p_est)
        expected_counts = expected_probs * sample_size
        expected_counts = expected_counts * (observed_counts.sum() / expected_counts.sum())
        
        # Combine bins with expected count < 5
        obs_binned, exp_binned = [], []
        obs_acc, exp_acc = 0, 0
        
        for obs, exp in zip(observed_counts, expected_counts):
            obs_acc += obs
            exp_acc += exp
            if exp_acc >= 5:
                obs_binned.append(obs_acc)
                exp_binned.append(exp_acc)
                obs_acc, exp_acc = 0, 0
        
        if obs_acc > 0 or exp_acc > 0:
            if len(exp_binned) > 0:
                obs_binned[-1] += obs_acc
                exp_binned[-1] += exp_acc
            else:
                obs_binned.append(obs_acc)
                exp_binned.append(exp_acc)
        
        if len(obs_binned) >= 2:
            chi2, pval = stats.chisquare(obs_binned, exp_binned, ddof=2)
            rejections.append(pval < alpha)
        else:
            rejections.append(np.nan)
    
    valid_rejections = [r for r in rejections if not np.isnan(r)]
    rejection_rate = np.mean(valid_rejections)
    
    print(f"True parameters: n={n_true}, p={p_true}")
    print(f"Sample size: {sample_size}, Simulations: {n_simulations}")
    print(f"Rejection rate: {rejection_rate:.3f} (expected ~{alpha})")
    print(f"Valid tests: {len(valid_rejections)}/{n_simulations}")
    return rejection_rate

# Run the test
test_binomial_rejection_rate()

True parameters: n=50, p=0.3
Sample size: 100, Simulations: 200
Rejection rate: 0.065 (expected ~0.05)
Valid tests: 200/200


np.float64(0.065)